In [ ]:
import shutil
import zipfile
from google.colab import drive
import os
import torch
drive.mount('/content/drive')
src_folder = '/content/drive/MyDrive/Colab Notebooks/Xai-Archive.zip'
dest_folder = '/content'
os.makedirs(dest_folder, exist_ok=True)
# Copy the folder
shutil.copyfile(src_folder, dest_folder + '/Xai-Archive.zip')
with zipfile.ZipFile(dest_folder + '/Xai-Archive.zip', 'r') as zip_ref:
    zip_ref.extractall(dest_folder)

src_folder_data = '/content/drive/MyDrive/Colab Notebooks/CUB_200_2011.tgz'    
shutil.copyfile(src_folder_data, dest_folder + '/CUB_200_2011.tgz')
os.makedirs("data/CUB_200_2011", exist_ok=True)
!pip install -e .
!tar -xzf CUB_200_2011.tgz -C data/CUB_200_2011

# ProtoPNet — CUB-200 Experiments (Member B)

Trains and evaluates **ProtoPNet** ("This Looks Like That", Chen et al. 2019) on
CUB-200-2011, with enriched metrics and the paper's "this looks like that"
prototype visualisations.

**ProtoPNet's three stages:**
1. **warm-up** — joint SGD with the backbone frozen (settles the random add-on +
   prototype layers onto the pretrained features);
2. **joint SGD** of all layers before the last layer;
3. **push** prototypes onto their nearest same-class patch, then **convex-optimise
   the last layer** (cross-entropy + L1).

**This notebook uses a single-cycle variant** (push once at the end, **no LR
decay**) with the backbone LR kept alive **+ stronger on-the-fly augmentation**.
Rationale: the paper tolerates a fast-decaying LR because it trains on a **~30×
offline-augmented** set (`img_aug.py`) we don't have — copying `step_size=5`
literally starves our backbone (the ~13% plateau). Keeping the LR alive + augmenting
on the fly lets the backbone actually fine-tune CUB. Details in the notes near the
bottom.

> Full training needs a GPU (Colab). On CPU/MPS set `SMOKE_TEST = True`.

In [1]:
import sys
sys.path.insert(0, '..')  # make src/ importable from notebooks directory

In [2]:
import time
import torch
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, Subset

from src.data import load_dataset
from src.data.transforms import get_transforms
from src.models import ProtoPNet, ProtoPNetTrainer
from src.evaluate import evaluate_model, print_results
from src.models.protopnet import (
    count_total_params,
    count_trainable_params,
    top_k_accuracy,
    mean_prototype_activation,
)
from src.visualize import (
    visualize_prototype_explanation,
    visualize_most_activated_prototypes,
)

In [3]:
# --- performance setup (Ampere+ GPU, e.g. your 22 GB card) ---
# These do NOT change results, only speed:
#  - TF32 matmul/conv: large speedup on Ampere, negligible numerical impact.
#  - cudnn.benchmark: autotunes kernels for our FIXED 224x224 input size.
if torch.cuda.is_available():
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    torch.backends.cudnn.benchmark = True
    torch.set_float32_matmul_precision('high')
    props = torch.cuda.get_device_properties(0)
    print(f'GPU: {props.name} | {props.total_memory / 1e9:.0f} GB VRAM')
else:
    print('No CUDA GPU detected — running on', 'mps' if torch.backends.mps.is_available() else 'cpu')

No CUDA GPU detected — running on mps


In [4]:
MODEL_NAME = 'protopnet'
DATASET = 'stanford_cars'           # 'stanford_cars' \ 'cub200'
BACKBONE = 'resnet34'        # 'resnet34' | 'vgg16'
NUM_CLASSES = 200

# Sweep parameter (README): prototypes per class in {5, 10, 20, 50}.
PROTOTYPES_PER_CLASS = 10
PROTOTYPE_DIM = 128

# --- training schedule: SINGLE-CYCLE, LATE LR decay (trainable backbone) ---
# The paper's step_size=5 froze our backbone early (-> 13%) because we lack its
# ~30x offline-augmented set. So we keep the LR alive long enough for the backbone
# to learn, then decay LATE to let val settle. Because we push only ONCE (at the
# end), late decay is safe — there's no post-push joint training to diverge.
WARM_EPOCHS = 5
JOINT_EPOCHS = 120        # single-cycle => ALL of these train the backbone
LAST_LAYER_ITERS = 40     # last-layer convex opt after the final push; 40 (not 20)
                          # so the classifier fully re-fits the pushed prototypes
                          # (20 left val_loss high -> ~6pt push penalty in exp7)
PUSH_INTERVAL = 10_000    # >> JOINT_EPOCHS => push only once, at the very end
SAVE_EVERY = 20           # write a *_latest recovery checkpoint every N joint epochs

# LATE LR decay: LR is constant for the first 70 joint epochs (backbone learns),
# then drops x0.1 so the last ~50 epochs settle. Safe (no mid-training push).
JOINT_LR_STEP_SIZE = 70
JOINT_LR_GAMMA = 0.1
JOINT_BACKBONE_LR = 3e-4   # backbone fine-tuning LR (paper 1e-4); higher to adapt
LAST_LR = 3e-4             # last-layer (classifier) LR (paper 1e-4); higher so the
                           # post-push re-fit recovers the pushed model faster

# Loss coefficients (paper coefs: crs_ent=1, clst=0.8, sep=-0.08, l1=1e-4).
CLUSTER_COEF = 0.8
SEPARATION_COEF = -0.08
L1_COEF = 1e-4

# Weight decay — paper main.py: 1e-3 on backbone + add-on only.
WEIGHT_DECAY = 1e-3

BATCH_SIZE = 80          # paper value; GPU had headroom (~4 GB/22 GB), so you can
                         # raise this (160-256) + scale LRs ~linearly for speed.
VAL_EVERY = 1
NUM_WORKERS = 8          # 50 GB RAM => more loader workers keep the GPU fed
USE_BBOX_CROP = True     # paper trains on bbox-cropped birds (cub200_cropped)
PERSISTANT_WORKERS = True
PREFETCH_FACTOR = 8

# Set True on a laptop to validate the pipeline on a tiny subset quickly.
SMOKE_TEST = False
SUBSET = 256

if torch.cuda.is_available():
    DEVICE = 'cuda'
elif torch.backends.mps.is_available():
    DEVICE = 'mps'
else:
    DEVICE = 'cpu'

if SMOKE_TEST:
    WARM_EPOCHS, JOINT_EPOCHS, LAST_LAYER_ITERS, SAVE_EVERY = 1, 2, 1, 1

print(f'Device: {DEVICE} | smoke_test={SMOKE_TEST}')

Device: mps | smoke_test=False


In [5]:
from torchvision import transforms
from src.data.transforms import IMAGENET_MEAN, IMAGENET_STD

train_ds = load_dataset(DATASET, 'train', use_bbox_crop=USE_BBOX_CROP)
val_ds = load_dataset(DATASET, 'val', use_bbox_crop=USE_BBOX_CROP)
test_ds = load_dataset(DATASET, 'test', use_bbox_crop=USE_BBOX_CROP)

# Stronger ProtoPNet-local TRAIN augmentation. We override the dataset's public
# .transform attribute (the same mechanism used for push_ds below) — NOT a change
# to Member A's shared transforms.py.
# IMPORTANT: the resize/crop matches the val/test pipeline (Resize(256) ->
# 224 crop) so train and eval see the SAME scale/aspect — earlier we used
# Resize((224,224)) which aspect-squashed train images and didn't match the
# Resize(256)+CenterCrop(224) eval, capping val accuracy. We keep rotation + shear
# (approx. the original img_aug.py geometric augs); a fresh random view per epoch
# stands in for their ~30x offline set.
train_ds.transform = transforms.Compose([
    transforms.Resize(256),
    transforms.RandomCrop(224),
    transforms.RandomAffine(degrees=15, shear=10),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.1),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

# Push must see CLEAN images: deterministic resize + center-crop, NO augmentation
# and NO shuffling, so prototypes anchor to real patches whose location is
# reproducible for visualisation. Reuse the train images with test transforms.
push_ds = load_dataset(DATASET, 'train', use_bbox_crop=USE_BBOX_CROP)
push_ds.transform = get_transforms('test', 224)

if SMOKE_TEST:
    train_ds = Subset(train_ds, range(min(SUBSET, len(train_ds))))
    val_ds = Subset(val_ds, range(min(SUBSET, len(val_ds))))
    test_ds = Subset(test_ds, range(min(SUBSET, len(test_ds))))
    push_ds = Subset(push_ds, range(min(SUBSET, len(push_ds))))

pin = (DEVICE == 'cuda')
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=pin, persistent_workers=PERSISTANT_WORKERS, prefetch_factor=PREFETCH_FACTOR)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=pin, persistent_workers=PERSISTANT_WORKERS, prefetch_factor=PREFETCH_FACTOR)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=pin, persistent_workers=PERSISTANT_WORKERS, prefetch_factor=PREFETCH_FACTOR)
# IMPORTANT: shuffle=False — source indices stored at push time must match push_ds.
push_loader = DataLoader(push_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=pin, persistent_workers=PERSISTANT_WORKERS, prefetch_factor=PREFETCH_FACTOR)

print(f'Train: {len(train_ds):,} | Val: {len(val_ds):,} | Test: {len(test_ds):,} | Push: {len(push_ds):,}')

Train: 6,516 | Val: 1,628 | Test: 8,041 | Push: 6,516


In [ ]:
model = ProtoPNet(
    backbone_name=BACKBONE,
    num_classes=NUM_CLASSES,
    num_prototypes_per_class=PROTOTYPES_PER_CLASS,
    prototype_dim=PROTOTYPE_DIM,
    cluster_coef=CLUSTER_COEF,
    separation_coef=SEPARATION_COEF,
    l1_coef=L1_COEF,
).to(DEVICE)

print(f'Prototypes: {model.num_prototypes:,} ({PROTOTYPES_PER_CLASS}/class)')
print(f'Parameters: {count_total_params(model):,} total')

trainer = ProtoPNetTrainer(
    model,
    device=DEVICE,
    warm_epochs=WARM_EPOCHS,
    joint_epochs=JOINT_EPOCHS,
    push_interval=PUSH_INTERVAL,
    last_layer_iters=LAST_LAYER_ITERS,
    l1_coef=L1_COEF,
    joint_backbone_lr=JOINT_BACKBONE_LR,
    last_lr=LAST_LR,
    joint_lr_step_size=JOINT_LR_STEP_SIZE,
    joint_lr_gamma=JOINT_LR_GAMMA,
    weight_decay=WEIGHT_DECAY,
)

In [ ]:
import os

# Best (pushed) model -> CKPT_PATH at the final push. Because this is a SINGLE-cycle
# run (one push, at the end), we also pass save_every so a *_latest recovery
# checkpoint is written every SAVE_EVERY joint epochs — interrupt-safe.
os.makedirs('../checkpoints', exist_ok=True)
CKPT_PATH = f'../checkpoints/{MODEL_NAME}_ppc{PROTOTYPES_PER_CLASS}_best.pth'

if DEVICE == 'cuda':
    torch.cuda.reset_peak_memory_stats()
start = time.perf_counter()

history = trainer.train(
    train_loader, val_loader,
    val_every=VAL_EVERY,
    push_loader=push_loader,
    checkpoint_path=CKPT_PATH,   # best (pushed) model, saved at the final push
    save_every=SAVE_EVERY,       # *_latest recovery checkpoint every SAVE_EVERY joint epochs
)

train_minutes = (time.perf_counter() - start) / 60
print(f'\nTraining wall-clock: {train_minutes:.1f} min')
print(f'Best post-push val acc: {trainer.best_val_acc:.4f}')
print(f'Best checkpoint: {CKPT_PATH}')
print(f'Recovery checkpoint: {CKPT_PATH.replace(".pth", "_latest.pth")}')
if DEVICE == 'cuda':
    print(f'Peak GPU memory: {torch.cuda.max_memory_allocated() / 1e6:.0f} MB')

In [ ]:
steps = list(range(1, len(history['train_loss']) + 1))

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# 'train total' carries the L1 term in EVERY phase (paper-faithful), so it sits
# on a large near-constant offset while the classifier is frozen. val loss is
# plain cross-entropy, so compare it against the 'train CE' component, not total.
axes[0].plot(steps, history['train_loss'], label='train total (incl. L1)')
if 'cls' in history:
    axes[0].plot(steps[:len(history['cls'])], history['cls'], label='train CE', linestyle='-.')
axes[0].plot(history['val_epochs'], history['val_loss'], label='val CE', linestyle='--')
for key in ('cluster', 'separation'):
    if key in history:
        axes[0].plot(steps[:len(history[key])], history[key], label=key, linestyle=':')
for i, p in enumerate(trainer.push_epochs):
    axes[0].axvline(p, color='green', linestyle=':', alpha=0.6, label='push' if i == 0 else None)
axes[0].set(title='ProtoPNet Loss (total has a ~const L1 offset — compare train CE vs val CE)',
            xlabel='Training epoch (incl. last-layer iters)', ylabel='Loss')
axes[0].legend()

axes[1].plot(steps, history['train_acc'], label='train')
axes[1].plot(history['val_epochs'], history['val_acc'], label='val', linestyle='--')
for i, p in enumerate(trainer.push_epochs):
    axes[1].axvline(p, color='green', linestyle=':', alpha=0.6, label='push' if i == 0 else None)
axes[1].set(title='ProtoPNet Accuracy', xlabel='Training epoch', ylabel='Top-1')
axes[1].legend()
plt.tight_layout()
os.makedirs('../results', exist_ok=True)
plt.savefig(f'../results/{MODEL_NAME}_training_curves.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# The best model was already saved to CKPT_PATH continuously during training
# (after every improvement), so this is optional — it just rewrites the same best
# checkpoint once the run completes. To recover an INTERRUPTED run instead, skip
# training and load CKPT_PATH into a freshly-built model:
#     ckpt = torch.load(CKPT_PATH, map_location=DEVICE, weights_only=False)
#     model.load_state_dict(ckpt['model_state'])
trainer.save_checkpoint(CKPT_PATH, epoch=len(history['train_loss']), history=history)
print(f'Best model saved to {CKPT_PATH} (val_acc={trainer.best_val_acc:.4f})')

## Evaluation & enriched metrics

Shared `evaluate_model` (top-1, per-class, inference time, FLOPs) plus the
ProtoPNet-specific enriched metrics required by the README.

In [ ]:
results = evaluate_model(model, test_loader, device=DEVICE)
print_results(results, model_name=MODEL_NAME)

per_class = results['per_class_accuracy']
top5 = top_k_accuracy(model, test_loader, device=DEVICE, k=5)
mean_act = mean_prototype_activation(model, test_loader, device=DEVICE)

print(f"\nTop-5 accuracy:              {top5:.4f}")
print(f"Per-class acc (mean +/- std): {per_class.mean():.4f} +/- {per_class.std():.4f}")
total_params = count_total_params(model)
currently_trainable = count_trainable_params(model)
print(f"Total parameters:            {total_params:,}")
print(f"Currently trainable params:  {currently_trainable:,}  # phase-dependent")
print(f"Number of prototypes:        {model.num_prototypes:,}")
print(f"Mean prototype activation:   {mean_act:.4f}")

In [ ]:
fig, ax = plt.subplots(figsize=(16, 4))
ax.bar(range(len(per_class)), sorted(per_class), width=1.0)
ax.axhline(per_class.mean(), color='red', linestyle='--', label=f'mean={per_class.mean():.3f}')
ax.set(title=f'{MODEL_NAME} Per-class accuracy (sorted)', xlabel='Class rank', ylabel='Accuracy')
ax.legend()
plt.tight_layout()
plt.savefig(f'../results/{MODEL_NAME}_per_class.png', dpi=150)
plt.show()

## Prototype visualisations — "this looks like that"

For three test images from different species, the top-k activated prototypes are
shown as **test image + heatmap + cyan box** (this) paired with the **source
training image + heatmap + yellow box** (that) — the training bird region each
prototype was pushed to.

In [ ]:
# Pick 3 test images from different species. Prefer correctly-classified examples
# for clean paper-style figures, but fall back to distinct species so the README
# deliverable always contains three qualitative visualisations.
seen, picks = set(), []
fallback_seen, fallback_picks = set(), []
base_test = test_ds.dataset if isinstance(test_ds, Subset) else test_ds
model.eval()
with torch.no_grad():
    for img, label in base_test:
        if label not in fallback_seen:
            fallback_seen.add(label)
            fallback_picks.append((img, label))
        if label in seen:
            continue
        pred = int(model(img.unsqueeze(0).to(DEVICE)).argmax(1).item())
        if pred == label:
            seen.add(label)
            picks.append((img, label))
        if len(picks) == 3 and len(fallback_picks) >= 3:
            break

if len(picks) < 3:
    picks = fallback_picks[:3]

for idx, (img, label) in enumerate(picks, start=1):
    fig = visualize_prototype_explanation(model, img, push_dataset=push_ds, top_k=3, device=DEVICE)
    fig.suptitle(f'Test image {idx} — class {label}', y=1.005, fontsize=14)
    plt.show()


In [ ]:
# Global view: the most-activated prototypes OVERALL (not restricted to the
# predicted class) — the paper's "most activated N prototypes" sanity check.
# On a healthy model the top global hits should belong to the predicted class.
img, label = picks[0]
fig = visualize_most_activated_prototypes(model, img, push_dataset=push_ds, top_k=5, device=DEVICE)
fig.suptitle(f'Most-activated prototypes (global) — class {label}', y=1.005, fontsize=14)
plt.show()

## Why single-cycle + late decay + augmentation (not the literal paper schedule)

The literal paper schedule (`step_size=5` decay, push every 10) gave us **~13%**:
the joint LR hits ~0 by epoch ~10, so the backbone barely fine-tunes. The paper
survives that because it trains on a **~30× offline-augmented** set (`img_aug.py`)
— its backbone learns a lot before the freeze. We don't have that set, so:

1. **Single cycle + late LR decay.** Push **once at the end** (no post-push joint
   training to diverge), and keep the LR **constant for the first
   `JOINT_LR_STEP_SIZE` joint epochs** (backbone fine-tunes), then drop it ×0.1 so
   the last epochs **settle** (smooths the val bounce a constant high LR causes).
   `joint_backbone_lr = 3e-4` (vs the paper's 1e-4) since it no longer decays early.
2. **Stronger, scale-matched on-the-fly augmentation** (rotation + shear + flip +
   jitter, in the data cell). The resize/crop **matches the val/test pipeline**
   (`Resize(256)` → 224 crop) so train and eval see the same scale — an earlier
   run used `Resize((224,224))`, which aspect-squashed train images and didn't
   match the eval transform, capping val accuracy.

**Progress across runs:** frozen backbone **0.13** → live LR (single-cycle) **~0.38**
→ + scale-matched aug + late decay (this config) — expected **~0.55–0.65**.

**Checkpoints.** Best **pushed** model → `CKPT_PATH` at the final push; a `*_latest`
recovery checkpoint every `SAVE_EVERY` joint epochs (may be un-pushed — run
`model.push_prototypes(push_loader, DEVICE)` for the interpretable version).

**Reading the loss curve.** L1 is in the loss in every phase, so the **total**
carries a near-constant offset while the classifier is frozen. Compare **train CE**
vs **val CE**, and watch **val accuracy**.

### Tips
- Watch the train-CE vs val-CE gap — if it keeps widening, augmentation/weight-decay
  is the lever (not more epochs).
- Free speedup: raise `BATCH_SIZE` to 160–256 (GPU had headroom) and scale LRs ~linearly.

## Hyperparameter sweep (Phase 3)

Re-run with `PROTOTYPES_PER_CLASS` ∈ {5, 10, 20, 50}, recording accuracy + the
enriched metrics for **every** value tested.

## Export README deliverables

The next cell writes all ProtoPNet artifacts required by the README into one local folder.
The following cell moves that folder to Google Drive.


In [ ]:
from pathlib import Path
import csv
import json
import shutil

import numpy as np

EXPORT_DIR = Path('../results') / f'{MODEL_NAME}_ppc{PROTOTYPES_PER_CLASS}_deliverables'
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

# Consolidated metrics required by the README.
metrics = {
    'method': MODEL_NAME,
    'dataset': DATASET,
    'backbone': BACKBONE,
    'prototypes_per_class': PROTOTYPES_PER_CLASS,
    'num_prototypes': model.num_prototypes,
    'prototype_dim': PROTOTYPE_DIM,
    'top1_accuracy': float(results['accuracy']),
    'top5_accuracy': float(top5),
    'per_class_accuracy_mean': float(per_class.mean()),
    'per_class_accuracy_std': float(per_class.std()),
    'training_time_min': float(train_minutes),
    'best_post_push_val_acc': float(trainer.best_val_acc),
    'peak_gpu_memory_mb': float(torch.cuda.max_memory_allocated() / 1024**2) if DEVICE == 'cuda' else 0.0,
    'total_parameters': int(count_total_params(model)),
    'currently_trainable_parameters': int(count_trainable_params(model)),
    'mean_prototype_activation': float(mean_act),
    'checkpoint_path': str(CKPT_PATH),
}

(EXPORT_DIR / 'metrics.json').write_text(json.dumps(metrics, indent=2))
with (EXPORT_DIR / 'metrics.csv').open('w', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=list(metrics))
    writer.writeheader()
    writer.writerow(metrics)

history_json = {key: [float(x) for x in value] for key, value in history.items() if isinstance(value, list)}
(EXPORT_DIR / 'training_history.json').write_text(json.dumps(history_json, indent=2))
np.save(EXPORT_DIR / 'per_class_accuracy.npy', per_class)

config = {
    'warm_epochs': WARM_EPOCHS,
    'joint_epochs': JOINT_EPOCHS,
    'last_layer_iters': LAST_LAYER_ITERS,
    'push_interval': PUSH_INTERVAL,
    'joint_lr_step_size': JOINT_LR_STEP_SIZE,
    'joint_lr_gamma': JOINT_LR_GAMMA,
    'joint_backbone_lr': JOINT_BACKBONE_LR,
    'last_lr': LAST_LR,
    'cluster_coef': CLUSTER_COEF,
    'separation_coef': SEPARATION_COEF,
    'l1_coef': L1_COEF,
    'weight_decay': WEIGHT_DECAY,
    'batch_size': BATCH_SIZE,
    'use_bbox_crop': USE_BBOX_CROP,
}
(EXPORT_DIR / 'run_config.json').write_text(json.dumps(config, indent=2))

# Training curves.
steps = list(range(1, len(history['train_loss']) + 1))
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(steps, history['train_loss'], label='train total')
if 'cls' in history:
    axes[0].plot(steps[:len(history['cls'])], history['cls'], label='train CE', linestyle='-.')
axes[0].plot(history['val_epochs'], history['val_loss'], label='val CE', linestyle='--')
for key in ('cluster', 'separation'):
    if key in history:
        axes[0].plot(steps[:len(history[key])], history[key], label=key, linestyle=':')
for i, p in enumerate(trainer.push_epochs):
    axes[0].axvline(p, color='green', linestyle=':', alpha=0.6, label='push' if i == 0 else None)
axes[0].set(title='ProtoPNet Loss', xlabel='Training epoch', ylabel='Loss')
axes[0].legend()
axes[1].plot(steps, history['train_acc'], label='train')
axes[1].plot(history['val_epochs'], history['val_acc'], label='val', linestyle='--')
for i, p in enumerate(trainer.push_epochs):
    axes[1].axvline(p, color='green', linestyle=':', alpha=0.6, label='push' if i == 0 else None)
axes[1].set(title='ProtoPNet Accuracy', xlabel='Training epoch', ylabel='Top-1')
axes[1].legend()
plt.tight_layout()
fig.savefig(EXPORT_DIR / 'training_curves.png', dpi=200, bbox_inches='tight')
plt.close(fig)

# Per-class accuracy plot.
fig, ax = plt.subplots(figsize=(16, 4))
ax.bar(range(len(per_class)), sorted(per_class), width=1.0)
ax.axhline(per_class.mean(), color='red', linestyle='--', label=f'mean={per_class.mean():.3f}')
ax.set(title='ProtoPNet per-class accuracy (sorted)', xlabel='Class rank', ylabel='Accuracy')
ax.legend()
plt.tight_layout()
fig.savefig(EXPORT_DIR / 'per_class_accuracy.png', dpi=200, bbox_inches='tight')
plt.close(fig)

# Qualitative prototype figures: three test images from different species.
for idx, (img, label) in enumerate(picks[:3], start=1):
    fig = visualize_prototype_explanation(model, img, push_dataset=push_ds, top_k=3, device=DEVICE)
    fig.suptitle(f'Test image {idx} - class {label}', y=1.005, fontsize=14)
    fig.savefig(EXPORT_DIR / f'this_looks_like_that_{idx}.png', dpi=200, bbox_inches='tight')
    plt.close(fig)

if picks:
    img, label = picks[0]
    fig = visualize_most_activated_prototypes(model, img, push_dataset=push_ds, top_k=5, device=DEVICE)
    fig.suptitle(f'Most-activated prototypes - class {label}', y=1.005, fontsize=14)
    fig.savefig(EXPORT_DIR / 'most_activated_prototypes.png', dpi=200, bbox_inches='tight')
    plt.close(fig)

# Copy the best checkpoint into the deliverables folder if it exists.
ckpt = Path(CKPT_PATH)
if ckpt.exists():
    shutil.copy2(ckpt, EXPORT_DIR / ckpt.name)

print(f'Exported ProtoPNet deliverables to: {EXPORT_DIR.resolve()}')
print('Files:')
for file in sorted(EXPORT_DIR.iterdir()):
    print(' -', file.name)


In [ ]:
from pathlib import Path
import shutil

DRIVE_EXPORT_ROOT = Path('/content/drive/MyDrive/xai-proto-vision/protopnet_exports')
if not Path('/content/drive').exists():
    from google.colab import drive
    drive.mount('/content/drive')

DRIVE_EXPORT_ROOT.mkdir(parents=True, exist_ok=True)
DRIVE_EXPORT_DIR = DRIVE_EXPORT_ROOT / EXPORT_DIR.name
if DRIVE_EXPORT_DIR.exists():
    shutil.rmtree(DRIVE_EXPORT_DIR)
shutil.move(str(EXPORT_DIR), str(DRIVE_EXPORT_DIR))

print(f'Moved deliverables to Google Drive: {DRIVE_EXPORT_DIR}')
